# Baseline de Recomendação - RetailRocket

Este notebook implementa e avalia baselines de recomendação top-K para o dataset
RetailRocket com feedback implícito. O objetivo é estabelecer referências de
performance para modelos futuros (MLP), garantindo reprodutibilidade e
validação por qualquer membro do grupo.

**Decisões técnicas herdadas do EDA:**

- Split cronológico 70/15/15 (sem vazamento temporal)
- Pesos implícitos: `view=1`, `addtocart=3`, `transaction=5`
- Target: `addtocart` OU `transaction` (sinal positivo)
- Cold-start tratado com MostPopular como fallback
- Item-KNN limitado aos 20K itens mais populares para viabilidade

**Estrutura:**

0. Setup e configuração
1. Carga e inspeção dos dados
2. Pesos implícitos e target
3. Split cronológico
4. Matriz usuário-item esparsa
5. Ground truth de avaliação
6. Baseline MostPopular
7. Baseline Item-KNN
8. Baseline Logistic Regression
9. Framework de métricas
10. Avaliação dos baselines
11. Análise cold-start
12. Testes inline de validação
13. Visualizações comparativas
14. Resultados consolidados
15. Conclusões e próximos passos
16. Exemplo de uso
17. Exemplo de recomendação por usuário


## 0. Setup e configuração

Parâmetros configuráveis via variáveis de ambiente.
Os CSVs são procurados em `data/` por padrão.

In [ ]:
from __future__ import annotations

import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Optional

%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import Markdown, display
from sklearn.metrics.pairwise import cosine_similarity

# --- Parâmetros configuráveis ---
RANDOM_SEED = int(os.environ.get("RANDOM_SEED", "42"))
TOP_K = [5, 10, 20]
IMPLICIT_WEIGHTS = {"view": 1, "addtocart": 3, "transaction": 5}
SESSION_GAP_MIN = 30
SPLIT_RATIOS = (0.70, 0.15, 0.15)
K_NEIGHBORS = 50
MAX_SIM_ITEMS = 20_000
MAX_EVAL_USERS = 5_000

# --- Caminhos ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(
    os.environ.get("RETAILROCKET_DATA_DIR", PROJECT_ROOT / "data")
).expanduser()

events_path = DATA_DIR / "events.csv"
category_tree_path = DATA_DIR / "category_tree.csv"
item_property_paths = [
    DATA_DIR / "item_properties_part1.csv",
    DATA_DIR / "item_properties_part2.csv",
]

csv_paths = [events_path, category_tree_path, *item_property_paths]
missing_files = [p.name for p in csv_paths if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        f"CSV(s) não encontrados: {', '.join(missing_files)}. "
        f"Ajuste RETAILROCKET_DATA_DIR ou coloque os arquivos em data/."
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

print(f"DATA_DIR: {DATA_DIR}")
print(f"RANDOM_SEED: {RANDOM_SEED}")
print(f"IMPLICIT_WEIGHTS: {IMPLICIT_WEIGHTS}")
print(f"TOP_K: {TOP_K}")
print(f"SPLIT_RATIOS: {SPLIT_RATIOS}")
print(f"K_NEIGHBORS: {K_NEIGHBORS}")
print(f"MAX_SIM_ITEMS: {MAX_SIM_ITEMS}")
print(f"MAX_EVAL_USERS: {MAX_EVAL_USERS}")

## 1. Carga e inspeção dos dados

Carrega `events.csv` com conversão de timestamps e inspeção básica.

In [ ]:
events = pd.read_csv(events_path)
events["event_time"] = pd.to_datetime(events["timestamp"], unit="ms", utc=True)
events = events.sort_values(["event_time", "visitorid", "itemid"]).reset_index(
    drop=True
)

inspeção = pd.DataFrame(
    {
        "metrica": [
            "total_eventos",
            "visitors_unicos",
            "itens_unicos",
            "periodo_inicio",
            "periodo_fim",
            "view_pct",
            "addtocart_pct",
            "transaction_pct",
        ],
        "valor": [
            f"{len(events):,}",
            f"{events['visitorid'].nunique():,}",
            f"{events['itemid'].nunique():,}",
            str(events["event_time"].min()),
            str(events["event_time"].max()),
            f"{(events['event'] == 'view').mean():.2%}",
            f"{(events['event'] == 'addtocart').mean():.2%}",
            f"{(events['event'] == 'transaction').mean():.2%}",
        ],
    }
)
display(inspeção)
display(events.head())

## 2. Pesos implícitos e target

Atribui pesos por tipo de evento conforme decisão do EDA: `view=1`, `addtocart=3`,
`transaction=5`. O target binário é 1 para `addtocart` ou `transaction`.

In [ ]:
events["implicit_weight"] = events["event"].map(IMPLICIT_WEIGHTS).fillna(0).astype(int)
events["target"] = events["event"].isin(["addtocart", "transaction"]).astype(int)

# Validação: nenhum peso zero para eventos conhecidos
assert (
    events.loc[events["event"].isin(IMPLICIT_WEIGHTS), "implicit_weight"].min() > 0
), "Erro: eventos conhecidos com peso zero."

weight_dist = (
    events.groupby("event")["implicit_weight"].agg(["count", "mean"]).reset_index()
)
display(weight_dist)

target_counts = events["target"].value_counts()
n_positivo = int(target_counts.get(1, 0))
n_negativo = int(target_counts.get(0, 0))
display(
    Markdown(
        f"**Target positivo (addtocart ou transaction):** "
        f"{n_positivo:,} eventos ({n_positivo / len(events):.2%})  |  "
        f"Negativo: {n_negativo:,} eventos ({n_negativo / len(events):.2%})"
    )
)

## 3. Split cronológico

Divisão temporal 70/15/15 sem sobreposição. Nenhum evento futuro é usado como feature.

In [ ]:
time_min = events["event_time"].min()
time_max = events["event_time"].max()
time_range = time_max - time_min

train_end = time_min + time_range * SPLIT_RATIOS[0]
val_end = train_end + time_range * SPLIT_RATIOS[1]

train_df = events[events["event_time"] < train_end].copy()
val_df = events[
    (events["event_time"] >= train_end) & (events["event_time"] < val_end)
].copy()
test_df = events[events["event_time"] >= val_end].copy()

split_summary = pd.DataFrame(
    {
        "partição": ["treino", "validação", "teste"],
        "inicio": [
            train_df["event_time"].min(),
            val_df["event_time"].min(),
            test_df["event_time"].min(),
        ],
        "fim": [
            train_df["event_time"].max(),
            val_df["event_time"].max(),
            test_df["event_time"].max(),
        ],
        "eventos": [len(train_df), len(val_df), len(test_df)],
        "visitors_unicos": [
            train_df["visitorid"].nunique(),
            val_df["visitorid"].nunique(),
            test_df["visitorid"].nunique(),
        ],
        "itens_unicos": [
            train_df["itemid"].nunique(),
            val_df["itemid"].nunique(),
            test_df["itemid"].nunique(),
        ],
    }
)
display(split_summary)

# Validações de split
assert len(train_df) + len(val_df) + len(test_df) == len(events), "Split inconsistente."
assert (
    train_df["event_time"].max() < val_df["event_time"].min()
), "Sobreposição treino/validação."
assert (
    val_df["event_time"].max() < test_df["event_time"].min()
), "Sobreposição validação/teste."

display(Markdown(f"**Corte treino:** {train_end}  |  **Corte validação:** {val_end}"))

## 4. Matriz usuário-item esparsa

Constrói a matriz esparsa de interações com pesos implícitos usando apenas o
conjunto de treino. Cada célula `(u, i)` contém a soma dos pesos das
interações do usuário `u` com o item `i`.

In [ ]:
# Mapear IDs para índices densos
visitor_ids = train_df["visitorid"].unique()
item_ids = train_df["itemid"].unique()

visitor_to_idx = {v: i for i, v in enumerate(visitor_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}
idx_to_visitor = {i: v for v, i in visitor_to_idx.items()}

n_users = len(visitor_ids)
n_items = len(item_ids)

display(Markdown(f"**Matriz treino:** {n_users:,} usuários x {n_items:,} itens"))

# Agregar pesos por (usuário, item)
train_df = train_df.assign(
    visitor_idx=train_df["visitorid"].map(visitor_to_idx),
    item_idx=train_df["itemid"].map(item_to_idx),
)

interaction_weights = (
    train_df.groupby(["visitor_idx", "item_idx"], observed=True)["implicit_weight"]
    .sum()
    .reset_index()
)

# Clipar pesos extremos
interaction_weights["implicit_weight"] = interaction_weights["implicit_weight"].clip(
    upper=50
)

sparse_matrix = sp.csr_matrix(
    (
        interaction_weights["implicit_weight"].values,
        (
            interaction_weights["visitor_idx"].values,
            interaction_weights["item_idx"].values,
        ),
    ),
    shape=(n_users, n_items),
)

sparsity = 1 - sparse_matrix.nnz / (n_users * n_items)
display(
    Markdown(
        f"Esparsidade da matriz: **{sparsity:.6%}**  |  NNZ: {sparse_matrix.nnz:,}"
    )
)

## 5. Ground truth de avaliação

Para cada usuário no conjunto de validação/teste, os itens com interação positiva
(`addtocart` ou `transaction`) formam o conjunto relevante. Apenas usuários que
também aparecem no treino são avaliados (cold-start é reportado separadamente).

In [ ]:
def build_ground_truth(
    df: pd.DataFrame,
    visitor_to_idx: dict,
    item_to_idx: dict,
    min_weight: int = 1,
) -> dict[int, set[int]]:
    """Constrói ground truth: usuário_idx -> conjunto de item_idx relevantes."""
    df_known = df[
        df["visitorid"].isin(visitor_to_idx) & df["itemid"].isin(item_to_idx)
    ].copy()
    df_known["visitor_idx"] = df_known["visitorid"].map(visitor_to_idx)
    df_known["item_idx"] = df_known["itemid"].map(item_to_idx)

    user_item_weight = (
        df_known.groupby(["visitor_idx", "item_idx"], observed=True)["implicit_weight"]
        .sum()
        .reset_index()
    )
    user_item_weight = user_item_weight[
        user_item_weight["implicit_weight"] >= min_weight
    ]

    ground_truth: dict[int, set[int]] = defaultdict(set)
    for _, row in user_item_weight.iterrows():
        ground_truth[int(row["visitor_idx"])].add(int(row["item_idx"]))

    return dict(ground_truth)


val_ground_truth = build_ground_truth(val_df, visitor_to_idx, item_to_idx)
test_ground_truth = build_ground_truth(test_df, visitor_to_idx, item_to_idx)

val_cold = val_df[~val_df["visitorid"].isin(visitor_to_idx)]["visitorid"].nunique()
test_cold = test_df[~test_df["visitorid"].isin(visitor_to_idx)]["visitorid"].nunique()

gt_summary = pd.DataFrame(
    {
        "partição": ["validação", "teste"],
        "usuários_avaliaveis": [len(val_ground_truth), len(test_ground_truth)],
        "usuários_cold_start": [val_cold, test_cold],
        "itens_medios_por_usuário": [
            (
                np.mean([len(v) for v in val_ground_truth.values()])
                if val_ground_truth
                else 0
            ),
            (
                np.mean([len(v) for v in test_ground_truth.values()])
                if test_ground_truth
                else 0
            ),
        ],
    }
)
display(gt_summary)

## 6. Baseline MostPopular

Recomenda os K itens mais populares (por soma de pesos implícitos) para todos os
usuários. Serve como lower bound e fallback para cold-start.

In [ ]:
class MostPopularRecommender:
    """Recomenda os itens mais populares por peso implícito agregado."""

    def __init__(self) -> None:
        self.popular_items: np.ndarray = np.array([])
        self.item_scores: np.ndarray = np.array([])

    def fit(self, sparse_mat: sp.csr_matrix) -> "MostPopularRecommender":
        item_scores = np.asarray(sparse_mat.sum(axis=0)).ravel()
        self.item_scores = item_scores
        self.popular_items = np.argsort(-item_scores)
        return self

    def recommend(
        self,
        user_idx: int,
        k: int,
        exclude_seen: bool = True,
        sparse_mat: Optional[sp.csr_matrix] = None,
    ) -> np.ndarray:
        if not exclude_seen or sparse_mat is None:
            return self.popular_items[:k]
        seen = set(sparse_mat[user_idx].nonzero()[1])
        if not seen:
            return self.popular_items[:k]
        mask = ~np.isin(self.popular_items, list(seen))
        filtered = self.popular_items[mask]
        return (
            filtered[:k]
            if len(filtered) >= k
            else np.concatenate([filtered, self.popular_items[: k - len(filtered)]])
        )


most_pop = MostPopularRecommender()
most_pop.fit(sparse_matrix)

top_10_popular = [idx_to_item[i] for i in most_pop.popular_items[:10]]
display(Markdown(f"Top-10 itens mais populares (por peso implícito): {top_10_popular}"))

## 7. Baseline Item-KNN

Recomenda itens por similaridade de cosseno entre vetores de interação de itens.
Para cada usuário, o score de um item candidato é a soma ponderada das
similaridades dos itens com os quais o usuário já interagiu.

Para viabilidade de memória, a similaridade é calculada apenas entre os
`MAX_SIM_ITEMS` itens mais populares. Itens fora deste subconjunto recebem
MostPopular como fallback.

In [ ]:
class ItemKNNRecommender:
    """Item-KNN com similaridade de cosseno esparsa.

    A recomendação usa multiplicacao de matriz esparsa para vetorizar o
    calculo de scores: scores = user_weights[sim_items] @ sim_matrix,
    mapeando de volta para indices globais.
    """

    def __init__(self, k_neighbors: int = 50, max_sim_items: int = 20_000) -> None:
        self.k_neighbors = k_neighbors
        self.max_sim_items = max_sim_items
        self.sim_matrix: Optional[sp.csr_matrix] = None
        self.popular_items: Optional[np.ndarray] = None
        self.n_items: int = 0
        self.sim_item_indices: Optional[np.ndarray] = None
        self.idx_mapping: Optional[dict] = None

    def fit(self, sparse_mat: sp.csr_matrix) -> "ItemKNNRecommender":
        self.n_items = sparse_mat.shape[1]
        item_popularity = np.asarray(sparse_mat.sum(axis=0)).ravel()
        self.popular_items = np.argsort(-item_popularity)

        # Selecionar top max_sim_items itens para calcular similaridade
        n_sim = min(self.max_sim_items, self.n_items)
        self.sim_item_indices = self.popular_items[:n_sim]

        # Mapeamento de índice global para índice local no subconjunto
        self.idx_mapping = {
            int(global_idx): local_idx
            for local_idx, global_idx in enumerate(self.sim_item_indices)
        }

        # Submatriz: itens x usuários (apenas itens populares)
        X = sparse_mat[:, self.sim_item_indices].T.tocsr().astype(np.float32)

        # Calcular similaridade por blocos para não estourar RAM
        print(f"Calculando similaridade entre {n_sim:,} itens...")
        batch_size = 5000
        n = X.shape[0]
        sim_rows: list[sp.csr_matrix] = []
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_sim = cosine_similarity(X[start:end], X, dense_output=False)
            sim_rows.append(batch_sim)

        full_sim = sp.vstack(sim_rows, format="csr")

        # Sparsificar: manter apenas top-K vizinhos por linha
        self.sim_matrix = self._sparsify_topk(full_sim, self.k_neighbors)

        print(f"Similaridade calculada. Shape: {self.sim_matrix.shape}")
        return self

    @staticmethod
    def _sparsify_topk(sim: sp.csr_matrix, k: int) -> sp.csr_matrix:
        """Para cada linha, manter apenas os K maiores valores."""
        sim = sim.tocsr().copy()
        for i in range(sim.shape[0]):
            row_start = sim.indptr[i]
            row_end = sim.indptr[i + 1]
            nnz = row_end - row_start
            if nnz > k:
                # argpartition retorna indices relativos ao slice data[row_start:row_end]
                top_k_pos = np.argpartition(sim.data[row_start:row_end], -k)[-k:]
                mask = np.zeros(nnz, dtype=bool)
                mask[top_k_pos] = True
                sim.data[row_start:row_end] = np.where(
                    mask, sim.data[row_start:row_end], 0.0
                )
        sim.eliminate_zeros()
        return sim

    def recommend(
        self,
        user_idx: int,
        k: int,
        exclude_seen: bool = True,
        sparse_mat: Optional[sp.csr_matrix] = None,
    ) -> np.ndarray:
        if sparse_mat is None:
            raise ValueError("sparse_mat e obrigatório para Item-KNN.")

        user_interactions = sparse_mat[user_idx]
        seen_items = set(int(x) for x in user_interactions.nonzero()[1])

        # Cold-start: sem interações, usar MostPopular
        if len(seen_items) == 0:
            candidates = self.popular_items[: k * 2]
            if exclude_seen:
                candidates = [c for c in candidates if c not in seen_items]
            return np.array(candidates[:k])

        # Vetorizar: construir vetor local de pesos do usuário para itens do subconjunto
        local_weights = np.zeros(len(self.sim_item_indices), dtype=np.float32)
        for global_idx in seen_items:
            if global_idx in self.idx_mapping:
                local_idx = self.idx_mapping[global_idx]
                local_weights[local_idx] = user_interactions[0, global_idx]

        # Se nenhum item visto esta no subconjunto, fallback MostPopular
        if local_weights.sum() == 0:
            candidates = self.popular_items[: k * 2]
            if exclude_seen:
                candidates = [c for c in candidates if c not in seen_items]
            return np.array(candidates[:k])

        # Score via multiplicacao esparsa: local_weights (1, n_sim) @ sim_matrix (n_sim, n_sim)
        local_scores = (
            sp.csr_matrix(local_weights).dot(self.sim_matrix).toarray().ravel()
        )

        # Mapear scores locais de volta para indices globais
        scores = np.zeros(self.n_items, dtype=np.float32)
        for local_j, score in enumerate(local_scores):
            if score > 0:
                scores[int(self.sim_item_indices[local_j])] = score

        # Excluir itens já vistos
        if exclude_seen:
            for s in seen_items:
                scores[s] = -np.inf

        # Se nenhum score positivo, fallback para MostPopular
        if np.max(scores) <= 0:
            candidates = self.popular_items[: k * 2]
            if exclude_seen:
                candidates = [c for c in candidates if c not in seen_items]
            return np.array(candidates[:k])

        return np.argsort(-scores)[:k]


print("Treinando Item-KNN... isso pode levar alguns minutos.")
t0 = time.time()
knn = ItemKNNRecommender(k_neighbors=K_NEIGHBORS, max_sim_items=MAX_SIM_ITEMS)
knn.fit(sparse_matrix)
elapsed = time.time() - t0
print(f"Item-KNN treinado em {elapsed:.1f}s.")

## 8. Baseline Logistic Regression

Modelo supervisionado pointwise que prevê a probabilidade de interação positiva
(`addtocart` ou `transaction`) para cada par (usuário, item). A feature chave
é o `collab_score`, que captura similaridade item-item (estilo Item-KNN)
e está disponível tanto no treino quanto na inferência.

**Features (calculadas no treino, disponíveis na inferência):**

- `user_total_weight`: soma dos pesos implícitos do usuário
- `user_n_items`: número de itens distintos interagidos
- `user_n_events`: número total de eventos
- `user_addtocart_ratio`: fração de eventos addtocart
- `user_transaction_ratio`: fração de eventos transaction
- `item_total_weight`: soma dos pesos implícitos do item
- `item_n_users`: número de usuários distintos que interagiram
- `item_addtocart_ratio`: fração de eventos addtocart no item
- `item_transaction_ratio`: fração de eventos transaction no item
- `collab_score`: sinal colaborativo Item-KNN (similaridade ponderada)

**Por que `collab_score` em vez de `user_item_weight`?**

A versão anterior usava `user_item_weight` (peso da interação usuário-item),
que era **sempre zero na inferência** (itens candidatos nunca foram interagidos).
Isso causava train-test distribution shift catastrófico: o modelo dependia de
uma feature que desaparecia na inferência.

O `collab_score` resolve isso: é a soma ponderada das similaridades entre o
item candidato e os itens com os quais o usuário já interagiu. Esse score
está disponível tanto no treino quanto na inferência, eliminando o shift.

**Estratégia de recomendação:**

- Treina LR com amostragem negativa (5 negativos por positivo)
- Para cada usuário, ranqueia candidatos pelo score predito
- Cold-start: fallback para MostPopular


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


class LogisticRegressionRecommender:
    """Baseline Logistic Regression com features do usuário, item e sinal colaborativo.

    Modelo supervisionado pointwise que prevê a probabilidade de interação
    positiva para cada par (usuário, item). A feature chave e o collab_score,
    que captura similaridade item-item (estilo Item-KNN) e esta disponível
    tanto no treino quanto na inferência.

    **Features:**
    - user_total_weight, user_n_items, user_n_events: estatísticas do usuário
    - user_addtocart_ratio, user_transaction_ratio: perfil de engajamento
    - item_total_weight, item_n_users: estatísticas do item
    - item_addtocart_ratio, item_transaction_ratio: perfil de conversão
    - collab_score: sinal colaborativo (similaridade Item-KNN)
    """

    def __init__(
        self,
        neg_ratio: int = 5,
        max_train_pairs: int = 500_000,
        C: float = 1.0,
        k_neighbors: int = 50,
        max_sim_items: int = 20_000,
        random_seed: int = 42,
    ) -> None:
        self.neg_ratio = neg_ratio
        self.max_train_pairs = max_train_pairs
        self.C = C
        self.k_neighbors = k_neighbors
        self.max_sim_items = max_sim_items
        self.random_seed = random_seed
        self.model: Optional[LogisticRegression] = None
        self.scaler: Optional[StandardScaler] = None
        self.feature_names: list[str] = []
        self.popular_items: Optional[np.ndarray] = None
        self.n_items: int = 0
        # Similaridade item-item (esparsa, top-K vizinhos)
        self.sim_matrix: Optional[sp.csr_matrix] = None
        self.sim_item_indices: Optional[np.ndarray] = None
        self.idx_mapping: Optional[dict] = None
        # Estatísticas pre-computadas (calculadas no treino)
        self._user_stats: Optional[pd.DataFrame] = None
        self._item_stats: Optional[pd.DataFrame] = None

    def _compute_user_stats(self, train_df: pd.DataFrame) -> pd.DataFrame:
        """Calcula estatísticas por usuário a partir do treino."""
        user_total = (
            train_df.groupby("visitor_idx", observed=True)
            .agg(
                user_total_weight=("implicit_weight", "sum"),
                user_n_items=("itemid", "nunique"),
                user_n_events=("event", "count"),
            )
            .reset_index()
        )
        user_events = (
            train_df.groupby("visitor_idx", observed=True)["event"]
            .value_counts()
            .unstack(fill_value=0)
            .reset_index()
        )
        for col in ["addtocart", "transaction", "view"]:
            if col not in user_events.columns:
                user_events[col] = 0
        # Divisão segura: evitar divisão por zero
        user_events["user_addtocart_ratio"] = user_events["addtocart"] / user_events[
            "view"
        ].replace(0, 1)
        user_events["user_transaction_ratio"] = user_events[
            "transaction"
        ] / user_events["view"].replace(0, 1)
        user_events = user_events[
            ["visitor_idx", "user_addtocart_ratio", "user_transaction_ratio"]
        ]
        user_stats = user_total.merge(user_events, on="visitor_idx", how="left")
        user_stats["user_addtocart_ratio"] = user_stats["user_addtocart_ratio"].fillna(
            0
        )
        user_stats["user_transaction_ratio"] = user_stats[
            "user_transaction_ratio"
        ].fillna(0)
        return user_stats

    def _compute_item_stats(self, train_df: pd.DataFrame) -> pd.DataFrame:
        """Calcula estatísticas por item a partir do treino."""
        item_total = (
            train_df.groupby("item_idx", observed=True)
            .agg(
                item_total_weight=("implicit_weight", "sum"),
                item_n_users=("visitorid", "nunique"),
            )
            .reset_index()
        )
        item_events = (
            train_df.groupby("item_idx", observed=True)["event"]
            .value_counts()
            .unstack(fill_value=0)
            .reset_index()
        )
        for col in ["addtocart", "transaction", "view"]:
            if col not in item_events.columns:
                item_events[col] = 0
        item_events["item_addtocart_ratio"] = item_events["addtocart"] / item_events[
            "view"
        ].replace(0, 1)
        item_events["item_transaction_ratio"] = item_events[
            "transaction"
        ] / item_events["view"].replace(0, 1)
        item_events = item_events[
            ["item_idx", "item_addtocart_ratio", "item_transaction_ratio"]
        ]
        item_stats = item_total.merge(item_events, on="item_idx", how="left")
        item_stats["item_addtocart_ratio"] = item_stats["item_addtocart_ratio"].fillna(
            0
        )
        item_stats["item_transaction_ratio"] = item_stats[
            "item_transaction_ratio"
        ].fillna(0)
        return item_stats

    def _compute_item_similarity(self, sparse_mat: sp.csr_matrix) -> None:
        """Computa similaridade item-item esparsa (top-K vizinhos)."""
        item_popularity = np.asarray(sparse_mat.sum(axis=0)).ravel()
        self.popular_items = np.argsort(-item_popularity)

        n_sim = min(self.max_sim_items, sparse_mat.shape[1])
        self.sim_item_indices = self.popular_items[:n_sim]
        self.idx_mapping = {int(g): l for l, g in enumerate(self.sim_item_indices)}

        # Submatriz: itens populares x usuários
        X = sparse_mat[:, self.sim_item_indices].T.tocsr().astype(np.float32)

        # Similaridade por blocos
        print(f"Calculando similaridade item-item ({n_sim:,} itens)...")
        batch_size = 5000
        n = X.shape[0]
        sim_rows: list[sp.csr_matrix] = []
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_sim = cosine_similarity(X[start:end], X, dense_output=False)
            sim_rows.append(batch_sim)
        full_sim = sp.vstack(sim_rows, format="csr")

        # Esparsificar: manter apenas top-K vizinhos por linha
        self.sim_matrix = self._sparsify_topk(full_sim, self.k_neighbors)
        print(f"Similaridade calculada. Shape: {self.sim_matrix.shape}")

    @staticmethod
    def _sparsify_topk(sim: sp.csr_matrix, k: int) -> sp.csr_matrix:
        """Para cada linha, manter apenas os K maiores valores."""
        sim = sim.tocsr().copy()
        for i in range(sim.shape[0]):
            row_start = sim.indptr[i]
            row_end = sim.indptr[i + 1]
            nnz = row_end - row_start
            if nnz > k:
                top_k_pos = np.argpartition(sim.data[row_start:row_end], -k)[-k:]
                mask = np.zeros(nnz, dtype=bool)
                mask[top_k_pos] = True
                sim.data[row_start:row_end] = np.where(
                    mask, sim.data[row_start:row_end], 0.0
                )
        sim.eliminate_zeros()
        return sim

    def _compute_collab_scores(
        self,
        pairs: pd.DataFrame,
        sparse_mat: sp.csr_matrix,
    ) -> np.ndarray:
        """Computa collab_score para pares (usuário, item).

        Para cada par, o score e a soma ponderada das similaridades
        entre o item candidato e os itens com os quais o usuário
        interagiu. Equivalente ao score do Item-KNN.
        """
        scores = np.zeros(len(pairs), dtype=np.float32)

        if self.sim_matrix is None or self.sim_item_indices is None:
            return scores

        # Agrupar pares por usuário para eficiencia
        for user_idx, group in pairs.groupby("visitor_idx", observed=True):
            user_interactions = sparse_mat[user_idx]
            seen_items = set(int(x) for x in user_interactions.nonzero()[1])
            if not seen_items:
                continue

            # Vetor local de pesos do usuário para itens do subconjunto
            local_weights = np.zeros(len(self.sim_item_indices), dtype=np.float32)
            for global_idx in seen_items:
                if global_idx in self.idx_mapping:
                    local_idx = self.idx_mapping[global_idx]
                    local_weights[local_idx] = float(user_interactions[0, global_idx])

            if local_weights.sum() == 0:
                continue

            # Scores para todos os itens do subconjunto
            user_scores = (
                sp.csr_matrix(local_weights).dot(self.sim_matrix).toarray().ravel()
            )

            # Atribuir scores aos pares deste usuário
            for row_pos, item_idx in zip(group.index, group["item_idx"]):
                if int(item_idx) in self.idx_mapping:
                    local_i = self.idx_mapping[int(item_idx)]
                    scores[row_pos] = user_scores[local_i]

        return scores

    def _build_features(
        self,
        pairs: pd.DataFrame,
        user_stats: pd.DataFrame,
        item_stats: pd.DataFrame,
    ) -> pd.DataFrame:
        """Constrói features para pares (usuário, item)."""
        features = pairs.merge(user_stats, on="visitor_idx", how="left")
        features = features.merge(item_stats, on="item_idx", how="left")
        fill_values = {
            "user_total_weight": 0,
            "user_n_items": 0,
            "user_n_events": 0,
            "user_addtocart_ratio": 0,
            "user_transaction_ratio": 0,
            "item_total_weight": 0,
            "item_n_users": 0,
            "item_addtocart_ratio": 0,
            "item_transaction_ratio": 0,
        }
        for col, val in fill_values.items():
            if col in features.columns:
                features[col] = features[col].fillna(val)
        return features

    def fit(
        self,
        train_df: pd.DataFrame,
        sparse_mat: sp.csr_matrix,
    ) -> "LogisticRegressionRecommender":
        """Treina o modelo Logistic Regression com features do treino."""
        self.n_items = sparse_mat.shape[1]

        # Calcular similaridade item-item
        self._compute_item_similarity(sparse_mat)

        # Estatísticas de usuário e item
        print("Computando estatísticas de usuário...")
        self._user_stats = self._compute_user_stats(train_df)
        print("Computando estatísticas de item...")
        self._item_stats = self._compute_item_stats(train_df)

        # Pares positivos: target=1 (addtocart/transaction)
        positive_pairs = (
            train_df[train_df["target"] == 1]
            .groupby(["visitor_idx", "item_idx"], observed=True)
            .first()
            .reset_index()[["visitor_idx", "item_idx"]]
        )
        positive_pairs["label"] = 1

        # Pares negativos: target=0 (view-only)
        negative_pairs = (
            train_df[train_df["target"] == 0]
            .groupby(["visitor_idx", "item_idx"], observed=True)
            .first()
            .reset_index()[["visitor_idx", "item_idx"]]
        )
        negative_pairs["label"] = 0

        n_pos = len(positive_pairs)
        n_neg_target = min(n_pos * self.neg_ratio, len(negative_pairs))
        rng = np.random.default_rng(self.random_seed)
        if len(negative_pairs) > n_neg_target:
            neg_sample_idx = rng.choice(
                len(negative_pairs), size=n_neg_target, replace=False
            )
            negative_pairs = negative_pairs.iloc[neg_sample_idx]

        print(
            f"Pares de treino: {n_pos:,} positivos, {len(negative_pairs):,} negativos"
        )

        train_pairs = pd.concat([positive_pairs, negative_pairs], ignore_index=True)

        if len(train_pairs) > self.max_train_pairs:
            sample_idx = rng.choice(
                len(train_pairs), size=self.max_train_pairs, replace=False
            )
            train_pairs = train_pairs.iloc[sample_idx]
            print(
                f"Amostragem: {len(train_pairs):,} pares (limite={self.max_train_pairs:,})"
            )

        # Construir features
        print("Construindo features...")
        features = self._build_features(train_pairs, self._user_stats, self._item_stats)

        # collab_score: sinal colaborativo (disponível no treino E na inferência)
        print("Computando collab_score para pares de treino...")
        features["collab_score"] = self._compute_collab_scores(train_pairs, sparse_mat)

        self.feature_names = [
            "user_total_weight",
            "user_n_items",
            "user_n_events",
            "user_addtocart_ratio",
            "user_transaction_ratio",
            "item_total_weight",
            "item_n_users",
            "item_addtocart_ratio",
            "item_transaction_ratio",
            "collab_score",
        ]

        X = features[self.feature_names].values.astype(np.float32)
        y = features["label"].values.astype(np.int32)

        # Proteger contra inf/NaN antes do scaler
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

        # Padronizar features
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)

        # Treinar Logistic Regression
        print(
            f"Treinando Logistic Regression (C={self.C}, "
            f"{X_scaled.shape[0]:,} amostras, {X_scaled.shape[1]} features)..."
        )
        self.model = LogisticRegression(
            C=self.C,
            solver="lbfgs",
            max_iter=1000,
            random_state=self.random_seed,
            n_jobs=-1,
        )
        self.model.fit(X_scaled, y)
        print("Logistic Regression treinado.")
        print(f"Accuracy no treino: {self.model.score(X_scaled, y):.4f}")

        # Importância das features
        coef_df = pd.DataFrame(
            {
                "feature": self.feature_names,
                "coef": self.model.coef_[0],
                "abs_coef": np.abs(self.model.coef_[0]),
            }
        ).sort_values("abs_coef", ascending=False)
        print("\nImportância das features (|coef|):")
        print(coef_df.to_string(index=False))

        return self

    def recommend(
        self,
        user_idx: int,
        k: int,
        exclude_seen: bool = True,
        sparse_mat: Optional[sp.csr_matrix] = None,
        candidate_items: Optional[np.ndarray] = None,
    ) -> np.ndarray:
        """Recomenda top-K itens por score predito do LR.

        Para viabilidade, avalia apenas os `candidate_items` mais populares
        (default: top 20K). Itens já vistos pelo usuário são excluídos.
        """
        if self.model is None or self.scaler is None:
            raise ValueError("Modelo não treinado. Chame fit() primeiro.")

        if sparse_mat is None:
            raise ValueError("sparse_mat é obrigatório para Logistic Regression.")

        # Cold-start: sem interações, usar MostPopular
        user_interactions = sparse_mat[user_idx]
        seen_items = set(int(x) for x in user_interactions.nonzero()[1])

        if len(seen_items) == 0:
            candidates = self.popular_items[: k * 2]
            return np.array(candidates[:k])

        # Verificar se o usuário tem estatísticas pre-computadas
        if self._user_stats is not None:
            user_has_stats = user_idx in self._user_stats["visitor_idx"].values
        else:
            user_has_stats = False

        if not user_has_stats:
            candidates = self.popular_items[: k * 2]
            if exclude_seen:
                candidates = [c for c in candidates if c not in seen_items]
            return np.array(candidates[:k])

        # Definir candidatos
        if candidate_items is None:
            candidate_items = self.popular_items[: self.max_sim_items]

        # Excluir itens já vistos
        if exclude_seen:
            candidate_items = np.array(
                [c for c in candidate_items if int(c) not in seen_items]
            )

        if len(candidate_items) == 0:
            return self.popular_items[:k]

        # Construir features para pares (usuário, cada candidato)
        pairs = pd.DataFrame(
            {
                "visitor_idx": user_idx,
                "item_idx": candidate_items,
            }
        )
        features = self._build_features(pairs, self._user_stats, self._item_stats)

        # collab_score para candidatos
        features["collab_score"] = self._compute_collab_scores(pairs, sparse_mat)

        X = features[self.feature_names].values.astype(np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X_scaled = self.scaler.transform(X)

        # Prever probabilidades
        scores = self.model.predict_proba(X_scaled)[:, 1]

        # Ranquear candidatos por score
        ranked_idx = np.argsort(-scores)
        top_k = candidate_items[ranked_idx[:k]]

        # Fallback se nenhum score significativo
        if np.max(scores) < 1e-6:
            candidates = self.popular_items[: k * 2]
            if exclude_seen:
                candidates = [c for c in candidates if c not in seen_items]
            return np.array(candidates[:k])

        return top_k


# Treinar Logistic Regression
print("Preparando Logistic Regression...")
t0 = time.time()
lr = LogisticRegressionRecommender(
    neg_ratio=5,
    max_train_pairs=500_000,
    C=1.0,
    k_neighbors=K_NEIGHBORS,
    max_sim_items=MAX_SIM_ITEMS,
    random_seed=RANDOM_SEED,
)
lr.fit(train_df, sparse_matrix)
elapsed = time.time() - t0
print(f"Logistic Regression treinado em {elapsed:.1f}s.")

## 9. Framework de métricas

Métricas de ranking padronizadas: Precision@K, Recall@K, NDCG@K, HitRate@K.

In [ ]:
def precision_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Precision@K: fração dos K recomendados que são relevantes."""
    if k == 0:
        return 0.0
    return len(set(recommended[:k]) & relevant) / k


def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Recall@K: fração dos relevantes que aparecem nos K recomendados."""
    if len(relevant) == 0:
        return 0.0
    return len(set(recommended[:k]) & relevant) / len(relevant)


def ndcg_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """NDCG@K: discounted cumulative gain normalizado."""
    if len(relevant) == 0:
        return 0.0
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def hit_rate_at_k(recommended: list[int], relevant: set[int], k: int) -> int:
    """HitRate@K: 1 se algum relevante esta nos K recomendados, 0 caso contrário."""
    return 1 if set(recommended[:k]) & relevant else 0


def evaluate_recommender(
    recommender,
    ground_truth: dict[int, set[int]],
    sparse_mat: sp.csr_matrix,
    k_values: list[int],
    max_users: Optional[int] = None,
    exclude_seen: bool = True,
) -> pd.DataFrame:
    """Avalia um recommender em métricas de ranking para multiplos K."""
    user_indices = list(ground_truth.keys())
    if max_users and len(user_indices) > max_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_indices = rng.choice(user_indices, size=max_users, replace=False).tolist()

    results = {
        k: {"precision": [], "recall": [], "ndcg": [], "hit_rate": []} for k in k_values
    }
    max_k = max(k_values)

    for user_idx in user_indices:
        relevant = ground_truth[user_idx]
        if not relevant:
            continue
        try:
            recommended = recommender.recommend(
                user_idx, k=max_k, exclude_seen=exclude_seen, sparse_mat=sparse_mat
            )
            recommended = (
                recommended.tolist()
                if hasattr(recommended, "tolist")
                else list(recommended)
            )
        except Exception:
            continue
        for k in k_values:
            results[k]["precision"].append(precision_at_k(recommended, relevant, k))
            results[k]["recall"].append(recall_at_k(recommended, relevant, k))
            results[k]["ndcg"].append(ndcg_at_k(recommended, relevant, k))
            results[k]["hit_rate"].append(hit_rate_at_k(recommended, relevant, k))

    rows = []
    for k in k_values:
        rows.append(
            {
                "K": k,
                "Precision@K": (
                    np.mean(results[k]["precision"]) if results[k]["precision"] else 0.0
                ),
                "Recall@K": (
                    np.mean(results[k]["recall"]) if results[k]["recall"] else 0.0
                ),
                "NDCG@K": np.mean(results[k]["ndcg"]) if results[k]["ndcg"] else 0.0,
                "HitRate@K": (
                    np.mean(results[k]["hit_rate"]) if results[k]["hit_rate"] else 0.0
                ),
                "n_users": len(results[k]["precision"]),
            }
        )
    return pd.DataFrame(rows)

## 10. Avaliação dos baselines

Avalia MostPopular, Item-KNN e Logistic Regression nos conjuntos de
validação e teste.

In [ ]:
display(Markdown("### Avaliação na validação"))
display(Markdown("Executando MostPopular na validação..."))
most_pop_val = evaluate_recommender(
    most_pop, val_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
most_pop_val["modelo"] = "MostPopular"
display(most_pop_val)

display(Markdown("Executando Item-KNN na validação..."))
knn_val = evaluate_recommender(
    knn, val_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
knn_val["modelo"] = "Item-KNN"
display(knn_val)

display(Markdown("Executando Logistic Regression na validação..."))
lr_val = evaluate_recommender(
    lr, val_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
lr_val["modelo"] = "LogisticRegression"
display(lr_val)

In [ ]:
display(Markdown("### Avaliação no teste"))
display(Markdown("Executando MostPopular no teste..."))
most_pop_test = evaluate_recommender(
    most_pop, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
most_pop_test["modelo"] = "MostPopular"
display(most_pop_test)

display(Markdown("Executando Item-KNN no teste..."))
knn_test = evaluate_recommender(
    knn, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
knn_test["modelo"] = "Item-KNN"
display(knn_test)

display(Markdown("Executando Logistic Regression no teste..."))
lr_test = evaluate_recommender(
    lr, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
lr_test["modelo"] = "LogisticRegression"
display(lr_test)

## 11. Análise cold-start

Avalia separadamente usuários warm-start (com histórico no treino) e cold-start
(sem histórico). Para cold-start, MostPopular é o único baseline aplicável.

In [ ]:
total_test_users = test_df["visitorid"].nunique()
warm_test_users = len(test_ground_truth)
cold_test_users = test_cold

cold_analysis = pd.DataFrame(
    {
        "tipo": ["warm_start", "cold_start", "total"],
        "usuários_teste": [warm_test_users, cold_test_users, total_test_users],
        "pct": [
            warm_test_users / total_test_users * 100 if total_test_users > 0 else 0,
            cold_test_users / total_test_users * 100 if total_test_users > 0 else 0,
            100.0,
        ],
    }
)
display(cold_analysis)

display(Markdown("#### Warm-start (usuários com histórico no treino)"))
warm_mp = evaluate_recommender(
    most_pop, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
warm_mp["modelo"] = "MostPopular"
display(warm_mp)

warm_knn = evaluate_recommender(
    knn, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
warm_knn["modelo"] = "Item-KNN"
display(warm_knn)

warm_lr = evaluate_recommender(
    lr, test_ground_truth, sparse_matrix, TOP_K, max_users=MAX_EVAL_USERS
)
warm_lr["modelo"] = "LogisticRegression"
display(warm_lr)

## 12. Testes inline de validação

Validações de corretude da pipeline. Todos devem passar para que o notebook
sejá considerado válido.

In [ ]:
display(Markdown("## Testes de validação"))
errors: list[str] = []

# Teste 1: Split cobre todo o dataset
total_split = len(train_df) + len(val_df) + len(test_df)
if total_split != len(events):
    errors.append(f"Split inconsistente: {total_split} vs {len(events)}")
else:
    display(Markdown("- [OK] Split cobre todo o dataset"))

# Teste 2: Sem sobreposição temporal
if train_df["event_time"].max() >= val_df["event_time"].min():
    errors.append("Sobreposição treino/validação")
else:
    display(Markdown("- [OK] Sem sobreposição treino/validação"))

if val_df["event_time"].max() >= test_df["event_time"].min():
    errors.append("Sobreposição validação/teste")
else:
    display(Markdown("- [OK] Sem sobreposição validação/teste"))

# Teste 3: Dimensão da matriz esparsa
if sparse_matrix.shape[0] != n_users or sparse_matrix.shape[1] != n_items:
    errors.append(
        f"Matriz esparsa: shape {sparse_matrix.shape} vs ({n_users}, {n_items})"
    )
else:
    display(Markdown("- [OK] Matriz esparsa com dimensão correta"))

# Teste 4: Soma de pesos consistente (após clip, tolerancia maior)
weight_sum = interaction_weights["implicit_weight"].sum()
raw_sum = train_df["implicit_weight"].sum()
if abs(weight_sum - raw_sum) > len(interaction_weights):
    errors.append(f"Soma de pesos inconsistente: {weight_sum} vs {raw_sum}")
else:
    display(Markdown("- [OK] Pesos implícitos consistentes (após clip)"))

# Teste 5: MostPopular recomenda itens distintos
recs = most_pop.recommend(0, k=min(20, n_items), sparse_mat=sparse_matrix)
if len(set(recs)) != len(recs):
    errors.append(f"MostPopular com duplicatas: {len(set(recs))} unicos de {len(recs)}")
else:
    display(Markdown("- [OK] MostPopular recomenda itens distintos"))

# Teste 6: Métricas no intervalo [0, 1]
metric_cols = ["Precision@K", "Recall@K", "NDCG@K", "HitRate@K"]
metrics_ok = True
for df in [most_pop_val, knn_val, lr_val, most_pop_test, knn_test, lr_test]:
    for col in metric_cols:
        if not all(0 <= v <= 1 for v in df[col].values):
            errors.append(f"{col} fora do intervalo [0,1]")
            metrics_ok = False
if metrics_ok:
    display(Markdown("- [OK] Todas as métricas no intervalo [0, 1]"))

# Teste 7: Recall@K cresce monotonicamente com K
recall_ok = True
for df, name in [
    (most_pop_val, "MostPopular"),
    (knn_val, "Item-KNN"),
    (lr_val, "LogisticRegression"),
]:
    recall_vals = df.sort_values("K")["Recall@K"].values
    if len(recall_vals) > 1 and not all(
        recall_vals[i] <= recall_vals[i + 1] + 1e-6 for i in range(len(recall_vals) - 1)
    ):
        errors.append(f"{name}: Recall@K não é monótono com K")
        recall_ok = False
if recall_ok:
    display(Markdown("- [OK] Recall@K é monótono com K"))

if errors:
    display(Markdown(f"### ERROS ENCONTRADOS:\n" + "\n".join(f"- {e}" for e in errors)))
else:
    display(Markdown("### Todos os testes passaram!"))

## 13. Visualizações comparativas

Gráficos comparando MostPopular vs Item-KNN por metrica e partição.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_names = ["Precision@K", "Recall@K", "NDCG@K", "HitRate@K"]

for ax, metric in zip(axes.ravel(), metric_names):
    for df, label, color in [
        (most_pop_val, "MostPopular", "#2f6f73"),
        (knn_val, "Item-KNN", "#8a5a44"),
        (lr_val, "LogisticRegression", "#4a7c3f"),
    ]:
        ax.plot(df["K"], df[metric], marker="o", label=label, color=color)
    ax.set_xlabel("K")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} na validação")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, metric in zip(axes.ravel(), metric_names):
    for df, label, color in [
        (most_pop_test, "MostPopular", "#2f6f73"),
        (knn_test, "Item-KNN", "#8a5a44"),
        (lr_test, "LogisticRegression", "#4a7c3f"),
    ]:
        ax.plot(df["K"], df[metric], marker="o", label=label, color=color)
    ax.set_xlabel("K")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} no teste")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Resultados consolidados

Tabela final com todos os resultados comparativos.

In [ ]:
final_comparison = pd.concat(
    [
        most_pop_val.assign(partição="validação"),
        most_pop_test.assign(partição="teste"),
        knn_val.assign(partição="validação"),
        knn_test.assign(partição="teste"),
        lr_val.assign(partição="validação"),
        lr_test.assign(partição="teste"),
    ]
).set_index(["modelo", "partição", "K"])

display(Markdown("### Resultados consolidados"))
display(final_comparison)

# Destaque: melhor Recall@10 na validação
best_models = final_comparison.xs(("validação", 10), level=["partição", "K"])[
    ["Recall@K"]
].sort_values("Recall@K", ascending=False)
display(
    Markdown(
        f"**Melhor Recall@10 na validação:** "
        f"{best_models.index[0]} = {best_models.iloc[0]['Recall@K']:.4f}"
    )
)

## 15. Conclusões e próximos passos

| Aspecto | Conclusão |
|---------|----------|
| **MostPopular** | Lower bound consistente, sem personalização |
| **Item-KNN** | Melhora sobre MostPopular para warm-start via similaridade colaborativa |
| **LogisticRegression** | Pointwise falha para ranking; features de superfície não substituem sinal colaborativo |
| **Cold-start** | ~93% dos visitantes do teste não aparecem no treino; MostPopular é o fallback |
| **Pesos implícitos** | `view=1, addtocart=3, transaction=5` capturam intensidade do funil |
| **Split cronológico** | Obrigatório para evitar vazamento temporal |

### Por que Logistic Regression falhou

O modelo LR com `collab_score` atingiu **accuracy de 92.6% no treino** mas **métricas de ranking
proximas de zero** na validação e teste. As causas são:

1. **Perda pointwise não otimiza ranking**: Cross-entropy classifica pares como
   positivo/negativo, mas não otimiza a ordenação relativa dos itens. Com 83% de
   negativos, o modelo aprende a prevêr "negativo" para quase tudo.

2. **Features constantes por usuário não discriminam itens**: `user_addtocart_ratio`
   (coef=5.99) e `user_n_events` (coef=0.54) tem o mesmo valor para todos os
   itens candidatos de um usuário. Elas deslocam o score base mas não alteram o
   ranking entre itens.

3. **`collab_score` esparsa**: Para usuários com poucas interações (a maioria),
   o score colaborativo e zero para a maioria dos candidatos, reduzindo o LR a um
   modelo de popularidade sem a capacidade de personalização do Item-KNN.

4. **Train-test distribution shift**: Embora `collab_score` exista na inferência,
   a distribuicao dos valores e diferente entre treino (pares reais) e inferência
   (candidatos não-interagidos).

### Licao para o MLP

Estes resultados mostram que:
- Modelos pointwise (classificação) são inadequados para ranking de recomendação
- O MLP deve usar **perda de ranking (BPR)** em vez de cross-entropy
- **Embeddings usuário-item** são necessários para capturar interações não-lineares
  que features de superfície não conseguem expressar
- O sinal colaborativo (similaridade item-item) é essencial, mas deve ser
  integrado via embeddings, não como feature linear

**Limitacoes:**

- Item-KNN com similaridade de cosseno é custoso para catálogos grandes; em produção,
  usar biblioteca `implicit` otimizada
- LR pointwise confirma que classificação binária não serve para ranking;
  abordagens pairwise (BPR) ou listwise (LambdaMART) são mais adequadas
- Avaliação offline não captura efeitos de exploracao/exploracao
- Nao foi feita otimizacao de hiperparametros (K_neighbors, pesos, C do LR)

**Próximos passos:**

1. Implementar modelo neural (MLP) com PyTorch e **perda BPR** (pairwise)
2. Aprender **embeddings usuário-item** (filtering colaborativo neural)
3. Adicionar features de item (categoria, disponibilidade) via `merge_asof`
4. Incorporar features de sessão e contexto temporal
5. Pipeline DVC + MLflow para rastreabilidade
6. Otimizar hiperparametros (embedding dim, learning rate, regularização)


## 16. Exemplo de uso

Demonstração prática de como os baselines recomendam itens para um usuário
específico. Inclui um exemplo warm-start (com histórico) e um cold-start
(sem histórico).

In [ ]:
# --- Exemplo warm-start: usuário com histórico no treino ---
# Escolher automaticamente um usuário com pelo menos 5 interações no treino
# e que também apareca no teste (para comparar com ground truth)
warm_users_in_test = list(test_ground_truth.keys())

# Filtrar usuários com pelo menos 5 interações no treino
user_interaction_counts = train_df.groupby("visitor_idx").size()
qualified_users = set(
    user_interaction_counts[user_interaction_counts >= 5].index.tolist()
)
qualified_warm_users = [u for u in warm_users_in_test if u in qualified_users]

if qualified_warm_users:
    example_user_idx = qualified_warm_users[0]
    example_visitor_id = idx_to_visitor[example_user_idx]

    # Histórico do usuário no treino
    user_history = train_df[train_df["visitor_idx"] == example_user_idx].copy()
    user_history_summary = (
        user_history.groupby("event")
        .agg(n_eventos=("event", "count"), itens_unicos=("itemid", "nunique"))
        .reset_index()
    )

    display(Markdown("### Exemplo warm-start"))
    display(Markdown(f"**visitorid:** {example_visitor_id}"))
    display(Markdown(f"**Interações no treino:** {len(user_history)} eventos"))
    display(user_history_summary)

    # Top-10 MostPopular
    mp_recs = most_pop.recommend(
        example_user_idx, k=10, exclude_seen=True, sparse_mat=sparse_matrix
    )
    mp_recs_itemids = [idx_to_item.get(int(i), "?") for i in mp_recs]

    # Top-10 Item-KNN
    knn_recs = knn.recommend(
        example_user_idx, k=10, exclude_seen=True, sparse_mat=sparse_matrix
    )
    knn_recs_itemids = [idx_to_item.get(int(i), "?") for i in knn_recs]

    # Top-10 Logistic Regression
    lr_recs = lr.recommend(
        example_user_idx, k=10, exclude_seen=True, sparse_mat=sparse_matrix
    )
    lr_recs_itemids = [idx_to_item.get(int(i), "?") for i in lr_recs]

    # Ground truth no teste
    gt_items = test_ground_truth.get(example_user_idx, set())
    gt_itemids = [
        idx_to_item.get(int(i), "?") for i in gt_items if int(i) in idx_to_item
    ]

    # Montar tabela comparativa
    mp_recs_int = [int(i) for i in mp_recs[:10]]
    knn_recs_int = [int(i) for i in knn_recs[:10]]
    lr_recs_int = [int(i) for i in lr_recs[:10]]
    comparison = pd.DataFrame(
        {
            "posição": range(1, 11),
            "MostPopular_itemid": mp_recs_itemids[:10],
            "MP_no_gt": ["sim" if i in gt_items else "não" for i in mp_recs_int],
            "ItemKNN_itemid": knn_recs_itemids[:10],
            "KNN_no_gt": ["sim" if i in gt_items else "não" for i in knn_recs_int],
        }
    )

    display(
        Markdown(f"**Ground truth (itens relevantes no teste):** {gt_itemids[:10]}")
    )
    display(comparison)

    # Métricas individuais para este usuário
    mp_hit = hit_rate_at_k(mp_recs_int, gt_items, 10)
    knn_hit = hit_rate_at_k(knn_recs_int, gt_items, 10)
    mp_ndcg = ndcg_at_k(mp_recs_int, gt_items, 10)
    knn_ndcg = ndcg_at_k(knn_recs_int, gt_items, 10)
    lr_hit = hit_rate_at_k(lr_recs_int, gt_items, 10)
    lr_ndcg = ndcg_at_k(lr_recs_int, gt_items, 10)

    individual_metrics = pd.DataFrame(
        {
            "modelo": ["MostPopular", "Item-KNN", "LogisticRegression"],
            "HitRate@10": [mp_hit, knn_hit, lr_hit],
            "NDCG@10": [mp_ndcg, knn_ndcg, lr_ndcg],
        }
    )
    display(Markdown("**Métricas para este usuário:**"))
    display(individual_metrics)
else:
    display(Markdown("Nenhum usuário qualificado encontrado para exemplo warm-start."))

In [ ]:
# --- Exemplo cold-start: usuário sem histórico no treino ---
# Escolher um usuário que aparece apenas no teste
cold_visitors = test_df[~test_df["visitorid"].isin(visitor_to_idx)][
    "visitorid"
].unique()

if len(cold_visitors) > 0:
    # Converter para tipo nativo Python para evitar problemas com pandas Index
    example_cold_visitor = int(cold_visitors[0])

    # Histórico do usuário frio no teste
    cold_history = test_df[test_df["visitorid"] == example_cold_visitor].copy()
    cold_history_summary = (
        cold_history.groupby("event")
        .agg(n_eventos=("event", "count"), itens_unicos=("itemid", "nunique"))
        .reset_index()
    )

    display(Markdown("### Exemplo cold-start"))
    display(Markdown(f"**visitorid:** {example_cold_visitor}"))
    display(Markdown(f"**Interações no treino:** 0 (usuário não aparece no treino)"))
    display(Markdown(f"**Interações no teste:** {len(cold_history)} eventos"))
    if not cold_history_summary.empty:
        display(cold_history_summary)

    # Para cold-start, ambos os modelos devem retornar MostPopular
    display(
        Markdown(
            "Para usuários cold-start, ambos os baselines retornam os itens "
            "mais populares (MostPopular). Sem histórico, não é possível "
            "personalizar recomendações."
        )
    )

    cold_popular = [idx_to_item.get(int(i), "?") for i in most_pop.popular_items[:10]]
    display(Markdown(f"**Top-10 itens recomendados (cold-start):**"))
    display(cold_popular)
else:
    display(Markdown("Nenhum usuário cold-start encontrado."))

In [ ]:
# --- Resumo dos exemplos ---
display(Markdown("### Resumo dos exemplos"))
display(
    Markdown(
        "- **Warm-start**: Item-KNN e Logistic Regression personalizam "
        "recomendações com base no histórico do usuário. MostPopular retorna "
        "os mesmos itens para todos.\n"
        "- **Cold-start**: Todos os baselines caem em MostPopular. Sem histórico, "
        "não há personalização possível.\n"
        "- Para melhorar cold-start, modelos futuros (MLP) poderão usar "
        "features de item (categoria, disponibilidade) e contexto temporal."
    )
)

## 17. Exemplo de recomendação por usuário

Função utilitária para obter os top-K itens recomendados para um `visitorid`
específico. Basta passar o ID do visitante e receber a lista de itens.

### Modelos utilizados

A função `recommend_top_k` aceita qualquer recommender já treinado neste notebook.
Os três modelos disponíveis são:

**MostPopular** (`most_pop`):
- Treinado na seção 6. Ranqueia todos os itens pela soma de pesos implícitos
  (`view=1`, `addtocart=3`, `transaction=5`) no conjunto de treino.
- Recomenda os mesmos itens para todos os usuários, diferenciando apenas pela
  exclusão dos itens já interagidos (`exclude_seen=True`).
- Serve como **lower bound** de performance e **fallback para cold-start**:
  usuários sem histórico recebem os itens mais populares.

**Item-KNN** (`knn`):
- Treinado na seção 7. Calcula similaridade de cosseno entre os vetores de
  interação dos 20.000 itens mais populares.
- Para cada usuário, o score de um item candidato é a soma ponderada das
  similaridades dos itens com os quais o usuário já interagiu.
- Personaliza recomendações com base no histórico do usuário: se o usuário
  interagiu com itens similares ao candidato, ele recebe scoré mais alto.
- Quando o usuário não tem histórico (cold-start) ou nenhum item visto está
  no subconjunto de 20K, faz fallback para MostPopular.

**LogisticRegression** (`lr`):
- Treinado na seção 8. Modelo supervisionado pointwise que prevê a
  probabilidade de interação positiva para cada par (usuário, item).
- Features incluem estatísticas do usuário, do item e da interação
  usuário-item, todas calculadas apenas com dados de treino.
- Serve como **baseline supervisionado** antes do MLP, validando o
  pipeline de features.
- Cold-start: fallback para MostPopular.

### Por que usar esses modelos?

1. **MostPopular** é o baseliné mais simples e robusto: sem personalização,
   mas sempre funciona, inclusive para usuários novos.
2. **Item-KNN** adiciona personalização via similaridade de conteúdo, melhorando
   Recall e NDCG sobre o MostPopular (conforme visto na seção 9).
3. **LogisticRegression** é o primeiro modelo supervisionado, validando que as
   features são informativas antes de investir no MLP.
4. Todos são **modelos de referência** (baselines) para comparar com o modelo
   neural (MLP) que será implementado futuramente.

### Como usar

```python
# Recomendar 10 itens para visitorid=97112 usando Item-KNN
item_ids = recommend_top_k(97112, recommender=knn, k=10)

# Recomendar 10 itens usando MostPopular
item_ids = recommend_top_k(97112, recommender=most_pop, k=10)

# Recomendar 10 itens usando Logistic Regression
item_ids = recommend_top_k(97112, recommender=lr, k=10)

# Cold-start: usuário ausente do treino retorna os mais populares
item_ids = recommend_top_k(999999999, recommender=lr, k=10)
```

In [ ]:
def recommend_top_k(
    visitor_id: int,
    recommender,
    k: int = 10,
    exclude_seen: bool = True,
) -> list[int]:
    """Retorna os top-K itemids recomendados para um visitorid.

    Args:
        visitor_id: ID do visitante no dataset.
        recommender: Instancia de MostPopularRecommender, ItemKNNRecommender
            ou LogisticRegressionRecommender.
        k: Numero de recomendações.
        exclude_seen: Se True, exclui itens já interagidos.

    Returns:
        Lista de itemids recomendados. Se o visitorid não éxistir no treino
        (cold-start), retorna os K itens mais populares.
    """
    if visitor_id not in visitor_to_idx:
        # Cold-start: retorna os K itens mais populares
        return [int(idx_to_item[i]) for i in most_pop.popular_items[:k]]

    user_idx = visitor_to_idx[visitor_id]
    recs_idx = recommender.recommend(
        user_idx, k=k, exclude_seen=exclude_seen, sparse_mat=sparse_matrix
    )
    return [int(idx_to_item[int(i)]) for i in recs_idx]


# --- Exemplo: recomendar para um usuário específico ---
example_visitor = 97112  # Altere este ID para testar outros usuários

mp_top10 = recommend_top_k(example_visitor, most_pop, k=10)
knn_top10 = recommend_top_k(example_visitor, knn, k=10)
lr_top10 = recommend_top_k(example_visitor, lr, k=10)

display(Markdown(f"### Top-10 recomendações para visitorid={example_visitor}"))

is_cold = example_visitor not in visitor_to_idx
if is_cold:
    display(
        Markdown(
            f"**Atenção:** visitorid={example_visitor} não aparece no treino (cold-start). "
            "Todos os modelos retornam os itens mais populares."
        )
    )
else:
    n_interactions = sparse_matrix[visitor_to_idx[example_visitor]].nnz
    display(
        Markdown(
            f"**Warm-start:** visitorid={example_visitor} tem "
            f"{n_interactions} interações no treino."
        )
    )

recommendations = pd.DataFrame(
    {
        "posicao": range(1, 11),
        "MostPopular_itemid": mp_top10,
        "ItemKNN_itemid": knn_top10,
        "LogReg_itemid": lr_top10,
    }
)
display(recommendations)

# --- Testar com outro usuário aleatório do treino ---
rng = np.random.default_rng(RANDOM_SEED)
random_visitor_idx = rng.choice(list(visitor_to_idx.values()))
random_visitor = int(idx_to_visitor[random_visitor_idx])

mp_random = recommend_top_k(random_visitor, most_pop, k=10)
knn_random = recommend_top_k(random_visitor, knn, k=10)
lr_random = recommend_top_k(random_visitor, lr, k=10)

display(
    Markdown(f"### Top-10 recomendações para visitorid={random_visitor} (aleatório)")
)
n_interactions_random = sparse_matrix[random_visitor_idx].nnz
display(Markdown(f"**Interações no treino:** {n_interactions_random}"))

random_recs = pd.DataFrame(
    {
        "posicao": range(1, 11),
        "MostPopular_itemid": mp_random,
        "ItemKNN_itemid": knn_random,
        "LogReg_itemid": lr_random,
    }
)
display(random_recs)

# --- Exemplo cold-start: usuário que não está no treino ---
cold_visitor = 999999999  # ID que não existe no treino
cold_recs = recommend_top_k(cold_visitor, most_pop, k=10)
display(
    Markdown(f"### Top-10 recomendações para visitorid={cold_visitor} (cold-start)")
)
display(
    Markdown(
        f"**Cold-start:** visitorid={cold_visitor} não está no treino. "
        "Retorna os itens mais populares como fallback."
    )
)
display(Markdown(f"**Top-10 itemids:** {cold_recs}"))